# 07_optuna_tune — LGBMRankerのハイパラ探索

`05_train_lgbm.ipynb` で作った5ソース×学習4週の特徴量テーブル(`training_data.py`で共通化した
OOM対策込みロード処理を再利用)に対して、optunaでLGBMRankerのハイパーパラメータを探索する。

候補プールのRecall上限(03の6ソース結合で0.093)に対し、現行モデルのMAP@12は0.0332とまだ距離があるため、
ハイパラ調整で今のモデルの実力をもう少し引き出せるか確認するのが狙い。

- 目的関数: valセットの**真の実購入集合**に対するMAP@12(候補プール外の購入も含む完全な正解集合)
- 学習は毎試行フルの300 estimatorsだと時間がかかりすぎるため、試行中はn_estimators上限を下げて
  早期終了(early stopping)で実質的な収束点を探す。ベストパラメータが決まった後に本番設定で学習し直す
- 試行数・時間はこのノートブック実行時のマシン負荷を見ながら`N_TRIALS`/`TIMEOUT_SEC`で調整する

In [ ]:
import os
import sys
import pickle
from datetime import datetime

import numpy as np
import pandas as pd
import lightgbm as lgb
import optuna

sys.path.append(os.path.abspath("."))
from common import MODELS_DIR, backtest_split, get_con, mapk, append_experiment_log
from training_data import group_sizes, load_train_val

optuna.logging.set_verbosity(optuna.logging.WARNING)

## データ読み込み(05と同じOOM対策込みロード)

In [ ]:
train_df, val_df, feature_cols = load_train_val(neg_group_sample_frac=0.15)
cat_cols = ["product_group_name", "index_group_name", "section_name"]

X_train, y_train = train_df[feature_cols], train_df["label"]
X_val, y_val = val_df[feature_cols], val_df["label"]

train_group_sizes = group_sizes(train_df, ["cutoff", "customer_idx"])
val_group_sizes = group_sizes(val_df, ["customer_idx"])

assert train_group_sizes.sum() == len(train_df)
assert val_group_sizes.sum() == len(val_df)
print(f"feature_cols ({len(feature_cols)})")
print(f"train groups: {len(train_group_sizes):,}, val groups: {len(val_group_sizes):,}")

In [ ]:
con = get_con()
cutoff_check, actual_map = backtest_split(con, holdout_days=7)
print(f"cutoff_check={cutoff_check} (val_dfのcutoffと一致するはず: {val_df['cutoff'].iloc[0]})")
print(f"実購入があった顧客数: {len(actual_map):,}")


def build_pred_map_from_scores(df, scores, k=12):
    tmp = df[["customer_idx", "article_id"]].copy()
    tmp["score"] = scores
    top = tmp.sort_values("score", ascending=False).groupby("customer_idx").head(k)
    return top.groupby("customer_idx")["article_id"].apply(list).to_dict()

## optunaの目的関数

num_leaves・learning_rate・min_child_samples・feature_fraction・bagging_fraction/freq・
lambda_l1/l2を探索する。試行中はn_estimators=150上限+early_stopping=20で高速化し、
MAP@12(真の正解集合に対する評価)を最大化する。

In [ ]:
N_TRIALS = 20
TIMEOUT_SEC = 3600  # 1時間を上限に、trial数と時間のどちらか早く達した方で打ち切る


def objective(trial):
    params = {
        "objective": "lambdarank",
        "n_estimators": 150,
        "num_leaves": trial.suggest_int("num_leaves", 15, 127),
        "learning_rate": trial.suggest_float("learning_rate", 0.02, 0.2, log=True),
        "min_child_samples": trial.suggest_int("min_child_samples", 10, 200),
        "feature_fraction": trial.suggest_float("feature_fraction", 0.5, 1.0),
        "bagging_fraction": trial.suggest_float("bagging_fraction", 0.5, 1.0),
        "bagging_freq": trial.suggest_int("bagging_freq", 0, 7),
        "lambda_l1": trial.suggest_float("lambda_l1", 1e-8, 10.0, log=True),
        "lambda_l2": trial.suggest_float("lambda_l2", 1e-8, 10.0, log=True),
        "random_state": 42,
        "verbosity": -1,
    }

    model = lgb.LGBMRanker(**params)
    model.fit(
        X_train, y_train,
        group=train_group_sizes,
        eval_set=[(X_val, y_val)],
        eval_group=[val_group_sizes],
        eval_at=[12],
        categorical_feature=cat_cols,
        callbacks=[lgb.early_stopping(20, verbose=False), lgb.log_evaluation(0)],
    )

    scores = model.predict(X_val)
    pred_map = build_pred_map_from_scores(val_df, scores)
    score = mapk(actual_map, pred_map, k=12)
    trial.set_user_attr("best_iteration", model.best_iteration_)
    return score


study = optuna.create_study(direction="maximize", sampler=optuna.samplers.TPESampler(seed=42))
study.optimize(objective, n_trials=N_TRIALS, timeout=TIMEOUT_SEC, show_progress_bar=True)

print(f"試行数: {len(study.trials)}")
print(f"ベストMAP@12(探索時n_estimators=150): {study.best_value:.4f}")
print(f"ベストパラメータ: {study.best_params}")

## 探索結果の確認

In [ ]:
trials_df = study.trials_dataframe().sort_values("value", ascending=False)
trials_df[["number", "value", "params_num_leaves", "params_learning_rate", "params_min_child_samples",
           "params_feature_fraction", "params_bagging_fraction", "params_bagging_freq",
           "params_lambda_l1", "params_lambda_l2"]].head(10)

## ベストパラメータで本番学習(n_estimators=500, early_stopping=30)

探索時より予算を増やして、ベストパラメータでの最終モデルを学習し直す。

In [ ]:
best_params = dict(study.best_params)
best_params.update({
    "objective": "lambdarank",
    "n_estimators": 500,
    "random_state": 42,
    "verbosity": -1,
})

final_ranker = lgb.LGBMRanker(**best_params)
final_ranker.fit(
    X_train, y_train,
    group=train_group_sizes,
    eval_set=[(X_val, y_val)],
    eval_group=[val_group_sizes],
    eval_at=[12],
    categorical_feature=cat_cols,
    callbacks=[lgb.early_stopping(30), lgb.log_evaluation(50)],
)

final_scores = final_ranker.predict(X_val)
final_pred_map = build_pred_map_from_scores(val_df, final_scores)
final_map12 = mapk(actual_map, final_pred_map, k=12)

results_df = pd.DataFrame([
    {"model": "optunaベストパラメータ(n_estimators=500)", "map@12": final_map12},
    {"model": "参考: 05(5ソース×4週、デフォルトパラメータ)", "map@12": 0.0332},
    {"model": "参考: 02_baselineハイブリッド", "map@12": 0.0248},
])
results_df

## モデル保存と実験ログ記録

In [ ]:
timestamp = datetime.now().strftime("%Y%m%d_%H%M")
ranker_path = os.path.join(MODELS_DIR, f"model_{timestamp}_ranker.pkl")
with open(ranker_path, "wb") as f:
    pickle.dump({"model": final_ranker, "feature_cols": feature_cols, "cat_cols": cat_cols}, f)
print(f"保存しました: {ranker_path}")

append_experiment_log(
    run_id=f"run_{timestamp}_ranker_optuna", timestamp=timestamp, notebook="07_optuna_tune.ipynb",
    description=f"LGBMRanker(LambdaRank)。optunaで{len(study.trials)}試行探索したベストパラメータで本番学習(n_estimators=500)",
    features=", ".join(feature_cols), params=str(best_params),
    cv_score=round(final_map12, 4), public_lb="", submission_file="", model_file=os.path.basename(ranker_path),
    notes=f"探索時ベストMAP@12(n_estimators=150)={study.best_value:.4f}。valは6候補ソース結合プール由来(学習は4週分)",
).tail(1)

## 次のステップ

- MAP@12が05(0.0332)を上回っていれば、`06_predict_submit.ipynb`をこのモデルで作り直しKaggleに提出する
- 上回らなければ、探索空間や試行数を見直すか、ここでハイパラ探索は一区切りとして別の方向(候補ソースFの統合、多期間検証など)を検討する